# SpLiCE Viz — Demo

Loads precomputed artifacts and runs example queries.

**Requirements:** `data/` must be populated (run `precompute.ipynb` first).

```
pip install numpy scipy faiss-cpu
```


In [1]:
# Colab Setup
import os
import sys

ON_COLAB = 'google.colab' in str(get_ipython())

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PATH = "/content/drive/Othercomputers/laptop/research"
else:
    possible_paths = [
        "../..",
        "/mnt/c/research",
        os.path.expanduser("~/research"),
    ]
    PATH = None
    for p in possible_paths:
        if os.path.exists(os.path.join(p, "tokens")):
            PATH = os.path.abspath(p)
            break
    if PATH is None:
        raise RuntimeError("Could not find research folder. Please set PATH manually.")

print(f"ON_COLAB: {ON_COLAB}")
print(f"PATH: {PATH}")

os.chdir(PATH)
sys.path.insert(0, PATH)

Mounted at /content/drive
ON_COLAB: True
PATH: /content/drive/Othercomputers/laptop/research


In [2]:
!pip install -q numpy scipy faiss-cpu ipywidgets requests pillow matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 94.1 MB/s eta 0:00:00


In [3]:
# Load all precomputed artifacts
import json
import numpy as np
import scipy.sparse
import faiss

DATA_PATH = os.path.join(PATH, "splice_viz", "data")

clip_embeddings     = np.load(os.path.join(DATA_PATH, "clip_embeddings.npy"))
concept_embeddings  = np.load(os.path.join(DATA_PATH, "concept_embeddings.npy"))
concept_frequencies = np.load(os.path.join(DATA_PATH, "concept_frequencies.npy"))
image_ids           = np.load(os.path.join(DATA_PATH, "image_ids.npy"))
splice_weights      = scipy.sparse.load_npz(os.path.join(DATA_PATH, "splice_weights.npz"))
faiss_index         = faiss.read_index(os.path.join(DATA_PATH, "faiss.index"))

with open(os.path.join(DATA_PATH, "vocabulary.json")) as f:
    vocab = json.load(f)
with open(os.path.join(DATA_PATH, "image_urls.json")) as f:
    image_urls = json.load(f)

vocab_index = {word: i for i, word in enumerate(vocab)}  # fast lookup

print(f"Images:    {clip_embeddings.shape[0]:,}")
print(f"Concepts:  {len(vocab):,}")
print(f"FAISS dim: {faiss_index.d}")

Images:    118,287
Concepts:  12,833
FAISS dim: 512


In [4]:
# Dataset statistics — computed once, referenced by helpers

splice_weights.eliminate_zeros()

# (3) Average concepts per image
avg_concepts_per_image = splice_weights.nnz / splice_weights.shape[0]
print(f"Avg concepts per image:  {avg_concepts_per_image:.1f}")

# Efficient sparse formats: CSR for row-slicing, CSC for column access in retrieve()
splice_csr = splice_weights.tocsr()
splice_csc  = splice_csr.tocsc()

# Per-concept fraction of images where it appears (for frequency enrichment ratio)
dataset_freq_per_concept = np.array((splice_csr > 0).mean(axis=0)).flatten()

# (2) Top-25% concepts by global mean weight — excluded from enrichment output
#     Removes ubiquitous background concepts whose enrichment is uninformative
top25_weight_threshold = np.percentile(concept_frequencies, 75)
top25pct_mask = concept_frequencies >= top25_weight_threshold
print(f"Top-25% weight threshold: {top25_weight_threshold:.4f}")
print(f"Excluded from enrichment: {top25pct_mask.sum():,} / {len(vocab):,} concepts")

Avg concepts per image:  16.1
Top-25% weight threshold: 0.0014
Excluded from enrichment: 3,220 / 12,833 concepts


In [5]:
# Query helpers — this is the logic the backend will run

def build_query_vector(concept_weights: dict) -> np.ndarray:
    """Weighted sum of concept CLIP embeddings, L2-normalised."""
    q = np.zeros(512, dtype=np.float32)
    missing = []
    for concept, weight in concept_weights.items():
        idx = vocab_index.get(concept)
        if idx is not None:
            q += weight * concept_embeddings[idx]
        else:
            missing.append(concept)
    if missing:
        print(f"  Not in vocabulary: {missing}")
    norm = np.linalg.norm(q)
    return q / norm if norm > 0 else q


def retrieve(concept_weights: dict, k: int = 1000, threshold: float = 0.22):
    """Top-k FAISS results filtered to cosine similarity >= threshold.

    Returns:
        display_results: all kept results as dicts (image_id, score, url, row)
        all_rows:        same row indices, for concept stats
    """
    q = build_query_vector(concept_weights)

    distances, indices = faiss_index.search(q[None], k)
    distances = distances[0]
    indices   = indices[0]

    mask       = distances >= threshold
    all_scores = distances[mask]
    all_idx    = indices[mask]

    display_results = []
    for score, idx in zip(all_scores, all_idx):
        img_id = int(image_ids[idx])
        display_results.append({
            "image_id": img_id,
            "score":    float(score),
            "url":      image_urls[str(img_id)],
            "row":      int(idx),
        })

    print(f"  {len(all_idx):,} / {k} results above threshold={threshold}")
    return display_results, list(all_idx)


def top_concepts(retrieved_rows: list[int], query_concepts: dict, top_n: int = 20):
    sub = splice_weights[retrieved_rows].toarray()  # [N, V]

    freq = (sub > 0).mean(axis=0)  # fraction of retrieved images with each concept

    for concept in query_concepts:
        if concept in vocab_index:
            freq[vocab_index[concept]] = 0.0
    freq[top25pct_mask] = 0.0

    top_idx = freq.argsort()[::-1][:top_n]
    top_idx = [i for i in top_idx if freq[i] > 0]

    out = []
    for i in top_idx:
        base_freq = float(dataset_freq_per_concept[i])
        out.append({
            "concept":        vocab[i],
            "freq_retrieved": float(freq[i]),
            "base_freq":      base_freq,
            "freq_ratio":     freq[i] / base_freq if base_freq > 0 else None,
        })

    return out

print("Helpers loaded.")

Helpers loaded.


In [6]:
import requests, matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

def show_results(results, n_cols=5):
    k = len(results)
    if k == 0:
        print('No results.')
        return
    n_rows = (k + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 3*n_rows))
    axes = np.array(axes).flatten()
    for ax, res in zip(axes, results):
        try:
            resp = requests.get(res['url'], timeout=5)
            img  = Image.open(BytesIO(resp.content)).convert('RGB')
            ax.imshow(img)
        except Exception:
            ax.set_facecolor('#eee')
        ax.set_title(f"{res['score']:.3f}", fontsize=8)
        ax.axis('off')
    for ax in axes[k:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def show_top_concepts(concepts):
    if not concepts:
        print('No concepts to display.')
        return
    hdr = f"{'Concept':25s}  {'FreqQ':>7s}  {'FreqDS':>7s}  {'FRatio':>7s}"
    print(hdr)
    print('-' * len(hdr))
    for c in concepts:
        fr = f"{c['freq_ratio']:.2f}x" if c['freq_ratio'] is not None else '   n/a'
        print(
            f"{c['concept']:25s}"
            f"  {c['freq_retrieved']*100:6.1f}%"
            f"  {c['base_freq']*100:6.1f}%"
            f"  {fr:>7s}"
        )

print('Display helpers loaded.')


Display helpers loaded.


In [14]:
import ipywidgets as widgets
from IPython.display import display, clear_output

QUERIES = {
    "woman":          {"woman": 1.0},
    "man":            {"man": 1.0},
    "cat + umbrella": {"cat": 0.3, "umbrella": 0.4},
    "nature":         {"barbecue": 1.0}
}

dropdown = widgets.Dropdown(options=list(QUERIES.keys()), description='Query:')
out = widgets.Output()

def run_query(change):
    with out:
        clear_output(wait=True)
        concept_weights = QUERIES[dropdown.value]
        results, all_rows = retrieve(concept_weights)
        show_results(results[:10])
        show_top_concepts(top_concepts(all_rows, concept_weights))

dropdown.observe(run_query, names='value')
display(dropdown, out)
run_query(None)


Dropdown(description='Query:', options=('woman', 'man', 'cat + umbrella', 'nature'), value='woman')

Output()